In [1]:
import numpy as np

In [ ]:
class MatrixFactorizationSGD:
    """
    Matrix Factorization with SGD — built directly on your SGDRegressor style
    Predicts rating = user_factor[u] dot item_factor[i]
    """
    def __init__(self, n_factors= 20, learning_rate=0.01, epochs=50, batch_size=1, reg=None, reg_param=0.0):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        self.P = None
        self.Q = None
        self.global_bias = 0.0
        

    def fit(self, user_ids, items_ids, ratings):
        """Fit the SGDRegressor to the training data"""
        # Convert to numpy arrays (safe for pandas DataFrames and Python list)
        user_ids = np.asarray(user_ids, dtype=int)
        items_ids = np.asarray(items_ids, dtype=int)
        ratings = np.asarray(ratings, dtype=float)
        
        # Determining the dimensions
        n_users = user_ids.max() + 1
        n_items = items_ids.max() + 1
        
        self.P = np.random.normal(0, 0.01, (n_users, self.n_factors)) # starting with random numbers drawn from a Gaussian distribution with mean 0 and SD 0.01
        self.Q = np.random.normal(0, 0.01, (n_items, self.n_factors))
        self.bias = np.mean(ratings)

        for _ in range(self.epochs):
            indices = np.random.permutation(len(ratings))

            for i in range(0, len(ratings), self.batch_size):
                batch_index = indices[i:i + self.batch_size]
                u_batch = user_ids[batch_index]
                i_batch = items_ids[batch_index]
                r_batch = ratings[batch_index]
        
                # Forward pass
                pred = np.sum(self.P[u_batch] * self.Q[i_batch], axis=1) + self.global_bias

                error = r_batch - pred

                # Gradients for Mean Squared Error (including the factor of 2)
                gradient_p = -2 * (error[:, np.newaxis] * [self.Q[i_batch]])
                gradient_q = -2 * (error[:, np.newaxis] * [self.P[u_batch]])

                if self.reg == 'l2':
                    gradient_p += 2 * self.reg_param * self.P[u_batch]
                    gradient_q += 2 * self.reg_param * self.Q[i_batch]
                elif self.reg == 'l1':
                    gradient_p += self.reg_param * np.sign(self.P[u_batch])
                    gradient_p += self.reg_param * np.sign(self.Q[i_batch])

                # SGD update (this is the exact rule from our earlier lesson!)
                self.P -= self.learning_rate * gradient_p
                self.Q -= self.learning_rate * gradient_q

        return self   

    def predict(self, user_ids, item_ids):
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        return np.sum(user_ids, item_ids, axis=1) + self.global_bias
 

   